**STAGE 1**

In [6]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, CODE], check=True)
print(subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
for _m in ["fyp_infra", "stage1_runner"]:
    sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
afb483d stage1: epoch sweep



wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs'}
GPU: NONE - switch runtime to GPU!


In [ ]:
#split size check
b, _ = sr.build_benchmark(sr.make_cfg(), paths)
print([len(e.dataset) for e in b.train_stream[:3]], [len(e.dataset) for e in b.test_stream[:3]])

In [7]:
EPOCH_GRID = [1, 5, 10, 20]   # edit if needed; 20 epochs × 20 tasks is the slow one, so run it last
CONFIGS = [sr.make_cfg(stage="stage1_epoch_sweep", strategy="naive", epochs=e, seed=0,
                       eval_split="val")   # tuning -> held-out validation slice, never test
           for e in EPOCH_GRID]
SHOW = ["epochs", "seed", "eval_split"]
for c in CONFIGS: print(run_id(c), {k: c[k] for k in SHOW})

b938bae18683 {'epochs': 1, 'seed': 0, 'eval_split': 'val'}
e44fdef7a7a2 {'epochs': 5, 'seed': 0, 'eval_split': 'val'}
714a54d5fe81 {'epochs': 10, 'seed': 0, 'eval_split': 'val'}
ac40efd77f20 {'epochs': 20, 'seed': 0, 'eval_split': 'val'}


In [8]:
out = run_sweep(CONFIGS, sr.run_experiment, paths)

[1/4] b938bae18683 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 40.00%  current 40.00%  train-loss 2.951
  exp 2/20: seen-avg 30.00%  current 60.00%  train-loss 2.695
  exp 3/20: seen-avg 12.40%  current 37.20%  train-loss 3.335
  exp 4/20: seen-avg 15.20%  current 60.80%  train-loss 2.971
  exp 5/20: seen-avg 7.76%  current 38.80%  train-loss 3.303
  exp 6/20: seen-avg 7.40%  current 44.40%  train-loss 3.157
  exp 7/20: seen-avg 8.34%  current 58.40%  train-loss 3.001
  exp 8/20: seen-avg 6.65%  current 53.20%  train-loss 3.079
  exp 9/20: seen-avg 4.98%  current 44.80%  train-loss 3.374
  exp 10/20: seen-avg 5.64%  current 56.40%  train-loss 3.262
  exp 11/20: seen-avg 5.24%  current 57.60%  train-loss 3.205
  exp 12/20: seen-avg 4.50%  current 54.00%  train-loss 3.337
  exp 13/20: seen-avg 3.38%  current 44.00%  train-loss 3.475
  exp 14/20: seen-avg 4.03%  current 56.40%  train-loss 3.188
  exp 15/20: seen-avg 3.47%  current 52.00%  train-loss 3.447
  exp 16/20: seen-avg 3.52%  current 56.40%  train-loss 3.075
  exp 17/20: 

acc_current_exp,▂█▁█▁▃▇▆▃▇▇▆▃▇▅▇▇▆▆▃
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[2/4] e44fdef7a7a2 running


...

acc_current_exp,▄█▁▆▃▂▇▇▃▅▆▃▃▅▆▇██▆▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[4/4] ac40efd77f20 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 74.00%  current 74.00%  train-loss 0.955
  exp 2/20: seen-avg 44.20%  current 88.40%  train-loss 0.671
  exp 3/20: seen-avg 25.07%  current 75.20%  train-loss 0.965
  exp 4/20: seen-avg 21.50%  current 86.00%  train-loss 0.723
  exp 5/20: seen-avg 15.76%  current 78.80%  train-loss 1.018
  exp 6/20: seen-avg 13.07%  current 78.40%  train-loss 0.928
  exp 7/20: seen-avg 12.06%  current 84.40%  train-loss 0.793
  exp 8/20: seen-avg 10.95%  current 87.60%  train-loss 0.645
  exp 9/20: seen-avg 8.80%  current 79.20%  train-loss 0.794
  exp 10/20: seen-avg 8.48%  current 84.80%  train-loss 0.745
  exp 11/20: seen-avg 7.13%  current 78.40%  train-loss 0.786
  exp 12/20: seen-avg 6.07%  current 72.80%  train-loss 0.842
  exp 13/20: seen-avg 6.22%  current 80.80%  train-loss 0.955
  exp 14/20: seen-avg 5.83%  current 81.60%  train-loss 0.750
  exp 15/20: seen-avg 5.49%  current 82.40%  train-loss 0.789
  exp 16/20: seen-avg 5.33%  current 85.20%  train-loss 0.645
  exp 17/

acc_current_exp,▁▇▂▆▃▃▆▇▄▆▃▁▄▅▅▆█▆▄▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


In [9]:
import numpy as np, pandas as pd
rows = []
for cfg in CONFIGS:
    r = load_result(cfg, paths)
    if r is None:
        rows.append({**{k: cfg[k] for k in SHOW}, "status": "not finished"}); continue
    m = r["metrics"]
    rows.append({**{k: cfg[k] for k in SHOW}, "run_id": r["run_id"],
                 "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
                 "avg_incremental_acc": m["avg_incremental_acc"],
                 "steps": m["optimizer_steps"], "n_params": m["n_params"], "minutes": r["wall_clock_s"] / 60})
df = pd.DataFrame(rows); df

,epochs,seed,eval_split,run_id,final_avg_acc,avg_forgetting,avg_incremental_acc,steps,n_params,minutes
0,1,0,val,b938bae18683,2.22,51.663158,8.705647,1420,1109240,20.436589
1,5,0,val,e44fdef7a7a2,3.62,68.673684,12.466774,7100,1109240,64.436245
2,10,0,val,714a54d5fe81,3.90,75.978947,13.468379,14200,1109240,117.254350
3,20,0,val,ac40efd77f20,4.28,81.810526,14.425892,28400,1109240,227.299411
